# Fine-Tuning DistilBERT for Stress Classification in Text

Train on the Dreaddit training split and evaluate the final checkpoint on the supplied test split.

In [ ]:
from datasets import load_dataset
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, set_seed

MODEL_NAME = "distilbert-base-uncased"
SEED = 42
set_seed(SEED)

dataset = load_dataset("andreagasparini/dreaddit")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=512)


tokenized = {}
for split in ("train", "test"):
    examples = dataset[split]
    columns_to_remove = [column for column in examples.column_names if column != "label"]
    tokenized[split] = examples.map(tokenize, batched=True, remove_columns=columns_to_remove)
    print(f"{split}: {len(tokenized[split])} examples")

## Train the classifier

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "NO_STRESS", 1: "STRESS"},
    label2id={"NO_STRESS": 0, "STRESS": 1},
)
args = TrainingArguments(
    output_dir="results",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.0,
    save_strategy="epoch",
    eval_strategy="no",
    seed=SEED,
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized["train"])
trainer.train()
trainer.save_model("results/final")
tokenizer.save_pretrained("results/final")

## Evaluate the final model

Precision, recall and F1 treat stress as the positive class.

In [ ]:
result = trainer.predict(tokenized["test"])
labels = result.label_ids
predictions = result.predictions.argmax(axis=1)
precision, recall, f1, _ = precision_recall_fscore_support(
    labels, predictions, average="binary", zero_division=0
)
tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
print(f"Accuracy: {accuracy_score(labels, predictions):.2%}")
print(f"Precision: {precision:.2%} | Recall: {recall:.2%} | F1: {f1:.2%}")
print(f"True negatives: {tn} | False positives: {fp} | False negatives: {fn} | True positives: {tp}")